In [1]:
import ee
import geopandas as gpd
import pandas as pd
from datetime import datetime

# Inicializa o GEE
ee.Initialize()

def media_nuvens_por_mes(shapefile_path, data_inicial, data_final, colecao, propriedade_nuvem='CLOUDY_PIXEL_PERCENTAGE', caminho_csv='media_nuvens.csv'):
    # Lê o shapefile e converte para GeoJSON
    gdf = gpd.read_file(shapefile_path)
    aoi = ee.Geometry.Polygon(gdf.geometry[0].__geo_interface__['coordinates'])

    # Converte datas
    data_inicio = ee.Date(data_inicial)
    data_fim = ee.Date(data_final)

    # Carrega coleção
    colecao_img = ee.ImageCollection(colecao).filterDate(data_inicio, data_fim).filterBounds(aoi)

    # Adiciona propriedades de ano e mês
    def adicionar_data(img):
        data = ee.Date(img.get('system:time_start'))
        return img.set({
            'ano': data.get('year'),
            'mes': data.get('month'),
            'nuvens': img.get(propriedade_nuvem)
        })

    colecao_com_data = colecao_img.map(adicionar_data)

    # Agrupa por ano e mês
    agrupado = colecao_com_data.aggregate_array('system:time_start').getInfo()
    lista_infos = colecao_com_data.aggregate_array([ 'ano', 'mes', 'nuvens' ]).getInfo()

    # Transforma para DataFrame
    df = pd.DataFrame(lista_infos, columns=['ano', 'mes', 'nuvens'])
    df['nuvens'] = pd.to_numeric(df['nuvens'], errors='coerce')

    # Agrupa por ano e mês e calcula média
    df_media = df.groupby(['ano', 'mes'], as_index=False)['nuvens'].mean()
    df_media.rename(columns={'nuvens': 'media_nuvens'}, inplace=True)

    # Salva em CSV
    df_media.to_csv(caminho_csv, index=False)
    print(f'Salvo em: {caminho_csv}')


EEException: Not signed up for Earth Engine or project is not registered. Visit https://developers.google.com/earth-engine/guides/access

In [ ]:
media_nuvens_por_mes(
    shapefile_path='sua_area.shp',
    data_inicial='2022-01-01',
    data_final='2023-12-31',
    colecao='COPERNICUS/S2_SR',
    propriedade_nuvem='CLOUDY_PIXEL_PERCENTAGE',
    caminho_csv='saida_nuvens.csv'
)
